# Experiment: claude_code

## 0 Setup

This section sets up the experiment environment:
- Review the variable dictionary to understand the dataset
- Set global random seed for reproducibility
- Verify required files are present

**Conventions:**
- All file paths are relative to this notebook's directory
- `random_state=42` is used wherever randomness is involved
- The notebook is designed to run sequentially from top to bottom without manual intervention

In [1]:
# Read and review the variable dictionary
DICT_PATH = "../../data/participation_2024-25_data_dictionary_cleaned.txt"

with open(DICT_PATH, "r") as f:
    dictionary_text = f.read()

print(dictionary_text)

UK Data Archive Data Dictionary

File-level information:

File Name = 		participation_2024-25_annual_data_open
Number of cases = 	34378


Variable-level information:

Pos. = 1,175	Variable = CARTS_NET	Variable label = In the last 12 months, engaged (attended OR participated) with the arts physically
This variable is    numeric, the SPSS measurement level is NOMINAL
SPSS user missing values = -1.7976931348623155e+308 thru -1.0
	Value label information for CARTS_NET
	Value = -3.0	Label = Not applicable
	Value = 1.0	Label = Yes
	Value = 2.0	Label = No
	Value = 3.0	Label = No & Missing

Pos. = 1,308	Variable = AGEBAND	Variable label = Respondent age band (ALL)
This variable is    numeric, the SPSS measurement level is SCALE
	Value label information for AGEBAND
	Value = -3.0	Label = Not Applicable
	Value = 1.0	Label = 16 to 19
	Value = 2.0	Label = 20 to 24
	Value = 3.0	Label = 25 to 29
	Value = 4.0	Label = 30 to 34
	Value = 5.0	Label = 35 to 39
	Value = 6.0	Label = 40 to 44
	Value = 7.0	Lab

In [2]:
# Set global random seed for reproducibility
import random
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

print(f"Random seed set to {RANDOM_STATE}")

Random seed set to 42


In [3]:
# Verify required files are present (do NOT load the data file)
import os

DATA_PATH = "../../data/participation_2024-25_experiment.tab"

files = {
    "Variable dictionary": DICT_PATH,
    "Data file": DATA_PATH,
}

for name, path in files.items():
    exists = os.path.exists(path)
    print(f"{name}: {path} -> {'FOUND' if exists else 'MISSING'}")
    assert exists, f"{name} not found at {path}"

Variable dictionary: ../../data/participation_2024-25_data_dictionary_cleaned.txt -> FOUND
Data file: ../../data/participation_2024-25_experiment.tab -> FOUND


# 1 Dataset Ingestion + Schema Checks + Problem Definition

## 1.1 Dataset Ingestion and Schema Checks

In this subsection we:
- Load the tab-separated data file into a pandas DataFrame
- Report the dataset dimensions (rows and columns)
- Verify that all 11 expected variables are present
- Check data types and value ranges against the variable dictionary

In [4]:
# 1.1 Read the data file
participation_raw = pd.read_csv(DATA_PATH, sep='\t')
print(f"Dataset shape: {participation_raw.shape[0]} rows x {participation_raw.shape[1]} columns")
print(f"\nColumn names: {list(participation_raw.columns)}")
print(f"\nData types:\n{participation_raw.dtypes}")
print(f"\nFirst 5 rows:")
participation_raw.head()

Dataset shape: 34378 rows x 11 columns

Column names: ['CARTS_NET', 'AGEBAND', 'SEX', 'QWORK', 'EDUCAT3', 'FINHARD', 'CINTOFT', 'gor', 'rur11cat', 'CHILDHH', 'COHAB']

Data types:
CARTS_NET    int64
AGEBAND      int64
SEX          int64
QWORK        int64
EDUCAT3      int64
FINHARD      int64
CINTOFT      int64
gor          int64
rur11cat     int64
CHILDHH      int64
COHAB        int64
dtype: object

First 5 rows:


,CARTS_NET,AGEBAND,SEX,QWORK,EDUCAT3,FINHARD,CINTOFT,gor,rur11cat,CHILDHH,COHAB
0,1,5,1,1,1,3,2,2,2,1,-3
1,1,6,2,2,1,4,2,2,2,1,-3
2,1,4,1,1,1,2,1,4,2,0,-3
3,1,8,1,2,1,1,1,9,1,0,-3
4,1,1,2,3,2,3,1,9,1,0,2


In [5]:
# Schema checks
EXPECTED_COLUMNS = ['CARTS_NET', 'AGEBAND', 'SEX', 'QWORK', 'EDUCAT3',
                    'FINHARD', 'CINTOFT', 'gor', 'rur11cat', 'CHILDHH', 'COHAB']

# Check 1: All expected columns present
missing_cols = set(EXPECTED_COLUMNS) - set(participation_raw.columns)
extra_cols = set(participation_raw.columns) - set(EXPECTED_COLUMNS)
print("Schema Check 1 — Column presence:")
print(f"  Expected columns: {len(EXPECTED_COLUMNS)}")
print(f"  Found columns:    {len(participation_raw.columns)}")
print(f"  Missing columns:  {missing_cols if missing_cols else 'None'}")
print(f"  Extra columns:    {extra_cols if extra_cols else 'None'}")
assert len(missing_cols) == 0, f"Missing columns: {missing_cols}"

# Check 2: All columns are numeric (as per variable dictionary)
print("\nSchema Check 2 — Data types:")
for col in EXPECTED_COLUMNS:
    dtype = participation_raw[col].dtype
    is_numeric = np.issubdtype(dtype, np.number)
    print(f"  {col:12s}: {str(dtype):10s} {'OK' if is_numeric else 'WARNING: expected numeric'}")

# Check 3: Value ranges
print("\nSchema Check 3 — Value ranges (min, max, unique count):")
for col in EXPECTED_COLUMNS:
    print(f"  {col:12s}: min={participation_raw[col].min():8.1f}, max={participation_raw[col].max():8.1f}, unique={participation_raw[col].nunique()}")

# Check 4: No fully-null columns
print("\nSchema Check 4 — NaN counts:")
nan_counts = participation_raw.isna().sum()
print(f"  Total NaN values across all columns: {nan_counts.sum()}")

Schema Check 1 — Column presence:
  Expected columns: 11
  Found columns:    11
  Missing columns:  None
  Extra columns:    None

Schema Check 2 — Data types:
  CARTS_NET   : int64      OK
  AGEBAND     : int64      OK
  SEX         : int64      OK
  QWORK       : int64      OK
  EDUCAT3     : int64      OK
  FINHARD     : int64      OK
  CINTOFT     : int64      OK
  gor         : int64      OK
  rur11cat    : int64      OK
  CHILDHH     : int64      OK
  COHAB       : int64      OK

Schema Check 3 — Value ranges (min, max, unique count):
  CARTS_NET   : min=     1.0, max=     3.0, unique=3
  AGEBAND     : min=    -3.0, max=   997.0, unique=17
  SEX         : min=    -5.0, max=   997.0, unique=5
  QWORK       : min=    -5.0, max=   999.0, unique=14
  EDUCAT3     : min=    -5.0, max=   999.0, unique=7
  FINHARD     : min=    -5.0, max=   997.0, unique=9
  CINTOFT     : min=    -5.0, max=     5.0, unique=8
  gor         : min=     1.0, max=     9.0, unique=9
  rur11cat    : min=     1.

### Schema Checks Performed

1. **Column presence**: Confirmed all 11 expected variables are present with no missing or extra columns.
2. **Data types**: Verified all columns are numeric, consistent with the variable dictionary.
3. **Value ranges**: Checked min/max values and unique counts for each variable to identify any out-of-range values.
4. **NaN counts**: Checked for any pandas-native NaN values (note: this dataset uses coded values like -3, 997, 999 for missing data rather than NaN).

## 1.2 Problem Definition

**Task**: Binary classification — predict whether a respondent engaged with the arts physically in the last 12 months.

**Framing**: We frame this as an **under-engagement identification problem** with social research value. Rather than treating arts participation as a pure personal preference, we investigate whether non-participation exhibits social patterns across demographic, socioeconomic, digital, and geographic dimensions. The goal is to identify population groups that may face structural or environmental barriers to physical arts engagement, supporting the development of more inclusive cultural policies.

**Target variable**: `CARTS_NET` — "In the last 12 months, engaged (attended OR participated) with the arts physically"
- Value 1 = Yes (engaged)
- Value 2 = No (did not engage)
- Values -3 (Not applicable) and 3 (No & Missing) will be **dropped in Step 2** to form a clean binary target. They are **not** dropped in this step.

**Feature variables** (10 predictors):

| Variable | Label | Type | Values |
|----------|-------|------|--------|
| `AGEBAND` | Respondent age band | Ordinal | 1 (16-19) to 15 (85+), 997 (Prefer not to say) |
| `SEX` | Respondent gender | Nominal | 1 (Female), 2 (Male), 997 (Prefer not to say) |
| `QWORK` | Current working status | Nominal | 1-10 (various statuses), 997/999 |
| `EDUCAT3` | Highest qualification | Nominal | 1 (Degree+), 2 (Other qualification), 997/999 |
| `FINHARD` | Financial hardship | Ordinal | 1 (Living comfortably) to 5 (Very difficult), 997 |
| `CINTOFT` | Internet usage frequency | Ordinal | 1 (Almost all the time) to 5 (Less often) |
| `gor` | Region (Government Office Region) | Nominal | 1-9 (English regions) |
| `rur11cat` | Rural/Urban area | Nominal | 1 (Rural), 2 (Urban) |
| `CHILDHH` | Children in household | Numeric/Ordinal | 0-3 numeric, 4 (4+), 997 |
| `COHAB` | Living as a couple | Nominal | 1 (Yes), 2 (No), 997 |

Negative codes (-3, -4, -5, -6) represent various forms of missing/invalid responses and will be addressed in the missingness handling step.

# 2 EDA and Insight Generation

In this section we:
- Convert the target variable to binary by removing missing/invalid rows
- Perform exploratory data analysis relevant to the prediction task
- Generate and save visualisations

## 2.1 Prepare Binary Target

In [6]:
# Remove rows where CARTS_NET is -3 or 3
print(f"Rows before target filtering: {len(participation_raw)}")
print(f"CARTS_NET value counts before filtering:")
print(participation_raw['CARTS_NET'].value_counts().sort_index())

mask = ~participation_raw['CARTS_NET'].isin([-3, 3])
participation_filtered = participation_raw[mask].copy()
print(f"\nRows after removing CARTS_NET = -3 and 3: {len(participation_filtered)}")
print(f"Rows removed: {len(participation_raw) - len(participation_filtered)}")

Rows before target filtering: 34378
CARTS_NET value counts before filtering:
CARTS_NET
1    31290
2     3048
3       40
Name: count, dtype: int64

Rows after removing CARTS_NET = -3 and 3: 34338
Rows removed: 40


In [7]:
# Create participation_eda: drop original CARTS_NET, add binary target
# Keep original coding: 1 = Yes (engaged), 2 = No (not engaged)
# Do NOT recode to 0/1 yet
participation_eda = participation_filtered.drop(columns=['CARTS_NET']).copy()
participation_eda['CARTS_NET_binary'] = participation_filtered['CARTS_NET'].values

print(f"participation_eda shape: {participation_eda.shape}")
print(f"\nTarget distribution (1=Yes/Engaged, 2=No/Not engaged):")
print(participation_eda['CARTS_NET_binary'].value_counts().sort_index())
print(f"\nTarget proportions:")
print(participation_eda['CARTS_NET_binary'].value_counts(normalize=True).sort_index().round(4))

participation_eda shape: (34338, 11)

Target distribution (1=Yes/Engaged, 2=No/Not engaged):
CARTS_NET_binary
1    31290
2     3048
Name: count, dtype: int64

Target proportions:
CARTS_NET_binary
1    0.9112
2    0.0888
Name: proportion, dtype: float64


In [8]:
# Create EDA figures folder
EDA_FOLDER = "evidence_claude_code/EDA_claude_code_Pics"
os.makedirs(EDA_FOLDER, exist_ok=True)
print(f"EDA figures folder created: {EDA_FOLDER}")

EDA figures folder created: evidence_claude_code/EDA_claude_code_Pics


## 2.2 Target Variable Distribution

In [9]:
# Target distribution
fig, ax = plt.subplots(figsize=(6, 4))
counts = participation_eda['CARTS_NET_binary'].value_counts().sort_index()
bars = ax.bar(['Yes (1)', 'No (2)'], counts.values, color=['#2ecc71', '#e74c3c'], edgecolor='black')
for bar, count in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
            f'{count}\n({count/len(participation_eda)*100:.1f}%)',
            ha='center', va='bottom', fontsize=10)
ax.set_title('Target Variable: Arts Engagement (CARTS_NET)', fontsize=12)
ax.set_ylabel('Count')
ax.set_xlabel('Engaged with arts physically?')
plt.tight_layout()
plt.savefig(f"{EDA_FOLDER}/target_distribution.png", dpi=150, bbox_inches='tight')
plt.show()

imbalance_ratio = counts.min() / counts.max()
print(f"\nClass imbalance ratio (minority/majority): {imbalance_ratio:.3f}")


Class imbalance ratio (minority/majority): 0.097


## 2.3 Feature Distributions

In [10]:
# Feature distributions — all features are categorical/ordinal
features = ['AGEBAND', 'SEX', 'QWORK', 'EDUCAT3', 'FINHARD',
            'CINTOFT', 'gor', 'rur11cat', 'CHILDHH', 'COHAB']

fig, axes = plt.subplots(2, 5, figsize=(22, 8))
axes = axes.flatten()

for i, feat in enumerate(features):
    vc = participation_eda[feat].value_counts().sort_index()
    axes[i].bar(vc.index.astype(str), vc.values, color='steelblue', edgecolor='black', alpha=0.8)
    axes[i].set_title(feat, fontsize=11, fontweight='bold')
    axes[i].tick_params(axis='x', rotation=45, labelsize=8)
    axes[i].set_ylabel('Count')

plt.suptitle('Feature Distributions (including coded missing values)', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(f"{EDA_FOLDER}/feature_distributions.png", dpi=150, bbox_inches='tight')
plt.show()

## 2.4 Feature-Target Relationships

In [11]:
# Feature-target relationships: engagement rate by feature value
# For each feature, compute the proportion of "Yes" (1) vs "No" (2) per category
fig, axes = plt.subplots(2, 5, figsize=(24, 10))
axes = axes.flatten()

for i, feat in enumerate(features):
    ct = pd.crosstab(participation_eda[feat], participation_eda['CARTS_NET_binary'], normalize='index')
    if 1 in ct.columns and 2 in ct.columns:
        ct = ct[[1, 2]]
        ct.columns = ['Engaged', 'Not engaged']
    ct.plot(kind='bar', stacked=True, ax=axes[i], color=['#2ecc71', '#e74c3c'],
            edgecolor='black', alpha=0.85, legend=(i == 0))
    axes[i].set_title(feat, fontsize=11, fontweight='bold')
    axes[i].set_ylabel('Proportion')
    axes[i].set_ylim(0, 1)
    axes[i].tick_params(axis='x', rotation=45, labelsize=8)

plt.suptitle('Engagement Rate by Feature Value (Stacked Proportions)', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(f"{EDA_FOLDER}/feature_target_relationships.png", dpi=150, bbox_inches='tight')
plt.show()

## 2.5 Correlation Analysis

In [12]:
# Correlation heatmap (Spearman — suitable for ordinal/coded data)
corr_matrix = participation_eda[features + ['CARTS_NET_binary']].corr(method='spearman')

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, ax=ax, square=True)
ax.set_title('Spearman Correlation Matrix (Features + Target)', fontsize=12)
plt.tight_layout()
plt.savefig(f"{EDA_FOLDER}/correlation_heatmap.png", dpi=150, bbox_inches='tight')
plt.show()

print("Correlations with target (CARTS_NET_binary):")
target_corr = corr_matrix['CARTS_NET_binary'].drop('CARTS_NET_binary').sort_values(key=abs, ascending=False)
print(target_corr.round(3))

Correlations with target (CARTS_NET_binary):
FINHARD     0.157
EDUCAT3    -0.143
SEX         0.088
CHILDHH     0.076
QWORK       0.065
CINTOFT    -0.060
COHAB       0.053
rur11cat    0.039
gor        -0.035
AGEBAND    -0.020
Name: CARTS_NET_binary, dtype: float64


## 2.6 Class Imbalance and Missing Value Overview

In [13]:
# Coded missing value prevalence per feature
# Negative codes (-3, -4, -5, -6) and codes 997, 999 represent missing/non-informative values
missing_codes = {-6, -5, -4, -3, 997, 999}

print("Coded missing/non-informative value counts per feature:")
print("=" * 60)
missing_summary = []
for feat in features:
    coded_missing = participation_eda[feat].isin(missing_codes).sum()
    pct = coded_missing / len(participation_eda) * 100
    missing_summary.append({'Feature': feat, 'Coded Missing': coded_missing, '% Missing': pct})
    print(f"  {feat:12s}: {coded_missing:6d} ({pct:5.2f}%)")

missing_df = pd.DataFrame(missing_summary)

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(missing_df['Feature'], missing_df['% Missing'], color='coral', edgecolor='black')
ax.set_xlabel('% Coded Missing/Non-informative Values')
ax.set_title('Prevalence of Coded Missing Values by Feature', fontsize=12)
for i, (pct, cnt) in enumerate(zip(missing_df['% Missing'], missing_df['Coded Missing'])):
    ax.text(pct + 0.3, i, f'{pct:.1f}% (n={cnt})', va='center', fontsize=9)
plt.tight_layout()
plt.savefig(f"{EDA_FOLDER}/missing_values_overview.png", dpi=150, bbox_inches='tight')
plt.show()

Coded missing/non-informative value counts per feature:
  AGEBAND     :    606 ( 1.76%)
  SEX         :    638 ( 1.86%)
  QWORK       :   1364 ( 3.97%)
  EDUCAT3     :   8239 (23.99%)
  FINHARD     :   1652 ( 4.81%)
  CINTOFT     :   2029 ( 5.91%)
  gor         :      0 ( 0.00%)
  rur11cat    :      0 ( 0.00%)
  CHILDHH     :     69 ( 0.20%)
  COHAB       :  24434 (71.16%)


## 2.7 EDA Summary and Key Insights

**Key observations for modelling:**

1. **Class imbalance**: The target variable shows an imbalanced distribution. This must be accounted for during train/test splitting (stratification) and metric selection (accuracy alone is insufficient).

2. **Feature-target patterns**: Several features show visible variation in engagement rates across categories, suggesting predictive signal. Age, education, financial hardship, and internet usage appear particularly relevant.

3. **Coded missing values**: Most features contain coded missing values (negative codes, 997, 999) that must be handled before modelling. The missing rates vary across features, requiring variable-specific treatment.

4. **All features are categorical/ordinal**: No continuous features are present. This affects preprocessing choices — one-hot encoding will be needed for nominal features (especially for Logistic Regression), while ordinal features may benefit from ordinal encoding.

5. **Sparse categories**: Some features have categories with very low counts (e.g., certain QWORK or AGEBAND values). These may need attention during encoding to avoid issues with rare categories in train/test splits.

6. **Moderate correlations**: Spearman correlations between features and target are generally moderate, suggesting no single feature dominates prediction. A multivariate model is appropriate.

# 3 Missingness Handling

In this section we handle coded missing values in the feature variables. Missing values in this dataset are not stored as `NaN` but are represented by negative codes and special values. The target variable has already been cleaned in Step 2.

## 3.1 Missingness Rules

Based on the variable dictionary, the following codes represent missing or non-informative responses:

| Code | Meaning |
|------|--------|
| -6 | Numeric response out of range (paper) |
| -5 | Multi-selected for single response (paper) |
| -4 | Not answered but should have (paper) |
| -3 | Not Answered / Not Applicable |
| 997 | Prefer not to say |
| 999 | Don't know |

### Variable-specific handling strategy

Naively dropping all rows with any coded missing value would cause catastrophic data loss (e.g., `COHAB` has ~71% coded missing). Instead we adopt a **variable-specific** strategy that balances data retention with modelling quality:

**Tier 1 — Low missingness (< 10%): Recode to NaN, then drop affected rows.**
Applies to: `AGEBAND`, `SEX`, `CHILDHH`

**Tier 2 — Moderate missingness (10–25%): Recode coded missing values to a separate "Unknown" category (code 0).**
Retaining these rows preserves sample size. The "Unknown" category may carry information (e.g., people who decline to answer may differ systematically).
Applies to: `QWORK`, `FINHARD`, `CINTOFT`, `EDUCAT3`

**Tier 3 — High missingness (> 25%): Recode to "Unknown" category (code 0).**
Dropping would eliminate the majority of the dataset.
Applies to: `COHAB` (~71% missing)

**No action needed**: `gor` and `rur11cat` have no coded missing values.

In all cases, negative codes (-6, -5, -4, -3) and special codes (997, 999) are treated as non-informative.

In [14]:
# 3.2 Apply missingness rules
print(f"Rows before missingness handling: {len(participation_eda)}")
print(f"Columns: {list(participation_eda.columns)}\n")

MISSING_CODES = [-6, -5, -4, -3, 997, 999]

# Classify features by missing rate
participation_temp = participation_eda.copy()
tier1_features = []  # low missing -> drop rows
tier2_features = []  # moderate/high missing -> recode to 0 (Unknown)

for feat in features:
    n_coded = participation_temp[feat].isin(MISSING_CODES).sum()
    pct = n_coded / len(participation_temp) * 100
    if n_coded == 0:
        print(f"  {feat:12s}: {n_coded:6d} coded missing ({pct:5.1f}%) -> No action needed")
    elif pct < 10:
        tier1_features.append(feat)
        print(f"  {feat:12s}: {n_coded:6d} coded missing ({pct:5.1f}%) -> Tier 1: will drop rows")
    else:
        tier2_features.append(feat)
        print(f"  {feat:12s}: {n_coded:6d} coded missing ({pct:5.1f}%) -> Tier 2/3: recode to Unknown (0)")

# Tier 2/3: Recode coded missing to 0 (Unknown category)
for feat in tier2_features:
    participation_temp.loc[participation_temp[feat].isin(MISSING_CODES), feat] = 0

# Tier 1: Recode coded missing to NaN, then drop those rows
for feat in tier1_features:
    participation_temp[feat] = participation_temp[feat].replace(MISSING_CODES, np.nan)

n_before_drop = len(participation_temp)
participation_clean = participation_temp.dropna(subset=tier1_features).copy()

print(f"\nTier 2/3 features recoded to Unknown (0): {tier2_features}")
print(f"Tier 1 features (rows dropped): {tier1_features}")
print(f"Rows dropped (Tier 1 NaN): {n_before_drop - len(participation_clean)} ({(n_before_drop - len(participation_clean))/n_before_drop*100:.1f}%)")
print(f"Rows remaining: {len(participation_clean)}")

Rows before missingness handling: 34338
Columns: ['AGEBAND', 'SEX', 'QWORK', 'EDUCAT3', 'FINHARD', 'CINTOFT', 'gor', 'rur11cat', 'CHILDHH', 'COHAB', 'CARTS_NET_binary']

  AGEBAND     :    606 coded missing (  1.8%) -> Tier 1: will drop rows
  SEX         :    638 coded missing (  1.9%) -> Tier 1: will drop rows
  QWORK       :   1364 coded missing (  4.0%) -> Tier 1: will drop rows
  EDUCAT3     :   8239 coded missing ( 24.0%) -> Tier 2/3: recode to Unknown (0)
  FINHARD     :   1652 coded missing (  4.8%) -> Tier 1: will drop rows
  CINTOFT     :   2029 coded missing (  5.9%) -> Tier 1: will drop rows
  gor         :      0 coded missing (  0.0%) -> No action needed
  rur11cat    :      0 coded missing (  0.0%) -> No action needed
  CHILDHH     :     69 coded missing (  0.2%) -> Tier 1: will drop rows
  COHAB       :  24434 coded missing ( 71.2%) -> Tier 2/3: recode to Unknown (0)

Tier 2/3 features recoded to Unknown (0): ['EDUCAT3', 'COHAB']
Tier 1 features (rows dropped): ['AGEBAN

In [15]:
# Verify: no remaining coded missing values, no NaN
print(f"Remaining NaN in features: {participation_clean[features].isna().sum().sum()}")
print(f"\nCoded missing value check (should all be 0):")
for feat in features:
    remaining = participation_clean[feat].isin(MISSING_CODES).sum()
    print(f"  {feat:12s}: {remaining}")

print(f"\nTarget distribution after cleaning:")
print(participation_clean['CARTS_NET_binary'].value_counts().sort_index())
print(f"\nTarget proportions:")
print(participation_clean['CARTS_NET_binary'].value_counts(normalize=True).sort_index().round(4))

Remaining NaN in features: 0

Coded missing value check (should all be 0):
  AGEBAND     : 0
  SEX         : 0
  QWORK       : 0
  EDUCAT3     : 0
  FINHARD     : 0
  CINTOFT     : 0
  gor         : 0
  rur11cat    : 0
  CHILDHH     : 0
  COHAB       : 0

Target distribution after cleaning:
CARTS_NET_binary
1    27763
2     2085
Name: count, dtype: int64

Target proportions:
CARTS_NET_binary
1    0.9301
2    0.0699
Name: proportion, dtype: float64


In [16]:
# Final verification of participation_clean
print("Final verification of participation_clean:")
print(f"  Shape: {participation_clean.shape}")
print(f"  NaN count: {participation_clean.isna().sum().sum()}")
print(f"\nValue ranges after cleaning:")
for feat in features:
    vals = sorted(participation_clean[feat].unique())
    print(f"  {feat:12s}: {vals}")

Final verification of participation_clean:
  Shape: (29848, 11)
  NaN count: 0

Value ranges after cleaning:
  AGEBAND     : [1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 11.0, 12.0, 13.0, 14.0, 15.0]
  SEX         : [1.0, 2.0]
  QWORK       : [1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 8.0, 9.0, 10.0]
  EDUCAT3     : [0, 1, 2]
  FINHARD     : [1.0, 2.0, 3.0, 4.0, 5.0]
  CINTOFT     : [1.0, 2.0, 3.0, 4.0, 5.0]
  gor         : [1, 2, 3, 4, 5, 6, 7, 8, 9]
  rur11cat    : [1, 2]
  CHILDHH     : [0.0, 1.0, 2.0, 3.0, 4.0]
  COHAB       : [0, 1, 2]


# 4 Baseline Model Training + Evaluation Harness

In this section we:
- Define X and y and prepare preprocessing pipelines
- Split data into train/validation/test sets (0.7/0.15/0.15)
- Create a common evaluation harness
- Train and evaluate a baseline Logistic Regression model

## 4.1 Prepare Modelling Data

In [17]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, average_precision_score,
                             confusion_matrix, classification_report, RocCurveDisplay,
                             PrecisionRecallDisplay)
import xgboost as xgb

# Define X and y
# Recode target: 1 (Yes/Engaged) -> 1, 2 (No/Not engaged) -> 0
X = participation_clean[features].copy()
y = (participation_clean['CARTS_NET_binary'] == 1).astype(int)  # 1=engaged, 0=not engaged

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"y distribution: {dict(y.value_counts().sort_index())}")
print(f"y=1 (engaged): {y.mean()*100:.1f}%,  y=0 (not engaged): {(1-y.mean())*100:.1f}%")

X shape: (29848, 10)
y shape: (29848,)
y distribution: {0: 2085, 1: 27763}
y=1 (engaged): 93.0%,  y=0 (not engaged): 7.0%


In [18]:
# Split: 0.7 train / 0.15 validation / 0.15 test, stratified
# First split: 70% train, 30% temp
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=RANDOM_STATE, stratify=y)

# Second split: 50/50 of temp -> 15% val, 15% test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=RANDOM_STATE, stratify=y_temp)

print(f"Train: {X_train.shape[0]} ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"Val:   {X_val.shape[0]} ({X_val.shape[0]/len(X)*100:.1f}%)")
print(f"Test:  {X_test.shape[0]} ({X_test.shape[0]/len(X)*100:.1f}%)")
print(f"\nTarget proportions (y=1):")
print(f"  Train: {y_train.mean():.4f}")
print(f"  Val:   {y_val.mean():.4f}")
print(f"  Test:  {y_test.mean():.4f}")

Train: 20893 (70.0%)
Val:   4477 (15.0%)
Test:  4478 (15.0%)

Target proportions (y=1):
  Train: 0.9302
  Val:   0.9301
  Test:  0.9301


In [19]:
# Preprocessing pipelines
# After missingness handling, some features now have a code-0 "Unknown" category
# All features are treated as categorical and one-hot encoded for both models

all_features = features  # all 10 feature columns

# Logistic Regression pipeline: one-hot encode + scale
lr_preprocessor = ColumnTransformer(
    transformers=[
        ('onehot', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='infrequent_if_exist'),
         all_features)
    ],
    remainder='drop'
)

# XGBoost pipeline: one-hot encode (no scaling needed)
xgb_preprocessor = ColumnTransformer(
    transformers=[
        ('onehot', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='infrequent_if_exist'),
         all_features)
    ],
    remainder='drop'
)

print("Preprocessing pipelines created:")
print("  LR:  OneHotEncoder (all features)")
print("  XGB: OneHotEncoder (all features)")

Preprocessing pipelines created:
  LR:  OneHotEncoder (all features)
  XGB: OneHotEncoder (all features)


## 4.2 Evaluation Harness

We define a common evaluation framework for all models. Given the task context (identifying under-engagement for policy purposes) and class imbalance, we include metrics that go beyond accuracy:

| Metric | Rationale |
|--------|-----------|
| **Accuracy** | Overall correctness, but can be misleading with imbalanced classes |
| **Precision** | Of those predicted as engaged, how many truly are — reduces false positives |
| **Recall** | Of those truly engaged, how many are identified — important for capturing all engaged individuals |
| **F1 Score** | Harmonic mean of precision and recall — balances both concerns |
| **ROC-AUC** | Discrimination ability across all thresholds — robust to class imbalance |
| **PR-AUC** | Precision-Recall AUC — especially informative when the positive class matters and classes are imbalanced |
| **Confusion Matrix** | Detailed view of prediction errors by class |

**Primary metric for model selection**: F1 Score (macro-averaged), as it balances precision and recall and is suitable for imbalanced binary classification where both classes matter. ROC-AUC serves as a secondary metric for overall discrimination.

In [20]:
def evaluate_model(model, X_eval, y_eval, model_name, dataset_name="Validation"):
    """Evaluate a model using the predefined evaluation harness."""
    y_pred = model.predict(X_eval)
    y_prob = model.predict_proba(X_eval)[:, 1]

    metrics = {
        'Accuracy': accuracy_score(y_eval, y_pred),
        'Precision (macro)': precision_score(y_eval, y_pred, average='macro'),
        'Recall (macro)': recall_score(y_eval, y_pred, average='macro'),
        'F1 (macro)': f1_score(y_eval, y_pred, average='macro'),
        'ROC-AUC': roc_auc_score(y_eval, y_prob),
        'PR-AUC': average_precision_score(y_eval, y_prob),
    }

    print(f"\n{'='*55}")
    print(f" {model_name} — {dataset_name} Set Results")
    print(f"{'='*55}")
    for metric, value in metrics.items():
        print(f"  {metric:25s}: {value:.4f}")

    print(f"\nConfusion Matrix:")
    cm = confusion_matrix(y_eval, y_pred)
    print(f"  Predicted ->    0 (No)  1 (Yes)")
    print(f"  Actual 0 (No):  {cm[0,0]:6d}  {cm[0,1]:6d}")
    print(f"  Actual 1 (Yes): {cm[1,0]:6d}  {cm[1,1]:6d}")

    print(f"\nClassification Report:")
    print(classification_report(y_eval, y_pred, target_names=['Not engaged (0)', 'Engaged (1)']))

    return metrics

print("Evaluation harness function defined.")

Evaluation harness function defined.


## 4.3 Baseline Model: Logistic Regression

In [21]:
# Train baseline Logistic Regression with standard hyperparameters
lr_baseline = Pipeline([
    ('preprocessor', lr_preprocessor),
    ('classifier', LogisticRegression(
        C=1.0,
        penalty='l2',
        solver='lbfgs',
        max_iter=1000,
        random_state=RANDOM_STATE,
        class_weight='balanced'  # handle imbalance
    ))
])

lr_baseline.fit(X_train, y_train)
print("Baseline Logistic Regression trained successfully.")
print(f"  Hyperparameters: C=1.0, penalty=l2, solver=lbfgs, class_weight=balanced")

Baseline Logistic Regression trained successfully.
  Hyperparameters: C=1.0, penalty=l2, solver=lbfgs, class_weight=balanced


In [22]:
# Evaluate baseline on VALIDATION set only
baseline_lr_metrics = evaluate_model(lr_baseline, X_val, y_val, "Baseline Logistic Regression", "Validation")


 Baseline Logistic Regression — Validation Set Results
  Accuracy                 : 0.7128
  Precision (macro)        : 0.5554
  Recall (macro)           : 0.6816
  F1 (macro)               : 0.5310
  ROC-AUC                  : 0.7475
  PR-AUC                   : 0.9722

Confusion Matrix:
  Predicted ->    0 (No)  1 (Yes)
  Actual 0 (No):     202     111
  Actual 1 (Yes):   1175    2989

Classification Report:
                 precision    recall  f1-score   support

Not engaged (0)       0.15      0.65      0.24       313
    Engaged (1)       0.96      0.72      0.82      4164

       accuracy                           0.71      4477
      macro avg       0.56      0.68      0.53      4477
   weighted avg       0.91      0.71      0.78      4477



# 5 Improving Performance

In this section we:
- Tune Logistic Regression on the validation set
- Train and tune XGBoost on the validation set
- Compare all models on the test set
- Select the final model using a multi-dimensional framework

## 5.1 Improve Logistic Regression

In [23]:
from sklearn.model_selection import GridSearchCV

# Define parameter grid for LR tuning
lr_param_grid = {
    'classifier__C': [0.001, 0.01, 0.1, 1.0, 10.0, 100.0],
    'classifier__penalty': ['l1', 'l2'],
    'classifier__solver': ['saga'],
    'classifier__class_weight': ['balanced', None],
}

lr_tuning_pipeline = Pipeline([
    ('preprocessor', lr_preprocessor),
    ('classifier', LogisticRegression(max_iter=2000, random_state=RANDOM_STATE))
])

# Use validation set for tuning via a custom approach:
# Combine train+val for GridSearchCV with a predefined split
from sklearn.model_selection import PredefinedSplit

X_train_val = pd.concat([X_train, X_val], axis=0)
y_train_val = pd.concat([y_train, y_val], axis=0)

# -1 for training samples, 0 for validation samples
test_fold = np.concatenate([-np.ones(len(X_train)), np.zeros(len(X_val))])
ps = PredefinedSplit(test_fold)

lr_grid = GridSearchCV(
    lr_tuning_pipeline,
    lr_param_grid,
    cv=ps,
    scoring='f1_macro',
    refit=True,
    n_jobs=-1,
    verbose=0
)

lr_grid.fit(X_train_val, y_train_val)

lr_tuned = lr_grid.best_estimator_
print("Logistic Regression tuning complete.")
print(f"  Best parameters: {lr_grid.best_params_}")
print(f"  Best validation F1 (macro): {lr_grid.best_score_:.4f}")

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The ma

Logistic Regression tuning complete.
  Best parameters: {'classifier__C': 0.1, 'classifier__class_weight': 'balanced', 'classifier__penalty': 'l1', 'classifier__solver': 'saga'}
  Best validation F1 (macro): 0.5468


In [24]:
# Structured tuning summary for Logistic Regression
print("=" * 60)
print(" TUNING SUMMARY: Logistic Regression")
print("=" * 60)
print(f"  Tuning method:          GridSearchCV with PredefinedSplit")
print(f"  Scoring metric:         F1 (macro)")
print(f"  Hyperparameters searched:")
print(f"    - C:                  {lr_param_grid['classifier__C']}")
print(f"    - penalty:            {lr_param_grid['classifier__penalty']}")
print(f"    - solver:             {lr_param_grid['classifier__solver']}")
print(f"    - class_weight:       {lr_param_grid['classifier__class_weight']}")
total_configs = (len(lr_param_grid['classifier__C']) *
                 len(lr_param_grid['classifier__penalty']) *
                 len(lr_param_grid['classifier__solver']) *
                 len(lr_param_grid['classifier__class_weight']))
print(f"  Total configurations:   {total_configs}")
print(f"  Iterations completed:   {total_configs}")
print(f"  Best hyperparameters:   {lr_grid.best_params_}")
print(f"  Best val F1 (macro):    {lr_grid.best_score_:.4f}")

 TUNING SUMMARY: Logistic Regression
  Tuning method:          GridSearchCV with PredefinedSplit
  Scoring metric:         F1 (macro)
  Hyperparameters searched:
    - C:                  [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]
    - penalty:            ['l1', 'l2']
    - solver:             ['saga']
    - class_weight:       ['balanced', None]
  Total configurations:   24
  Iterations completed:   24
  Best hyperparameters:   {'classifier__C': 0.1, 'classifier__class_weight': 'balanced', 'classifier__penalty': 'l1', 'classifier__solver': 'saga'}
  Best val F1 (macro):    0.5468


In [25]:
# Evaluate tuned LR on validation set
tuned_lr_val_metrics = evaluate_model(lr_tuned, X_val, y_val, "Tuned Logistic Regression", "Validation")


 Tuned Logistic Regression — Validation Set Results
  Accuracy                 : 0.6868
  Precision (macro)        : 0.5474
  Recall (macro)           : 0.6618
  F1 (macro)               : 0.5122
  ROC-AUC                  : 0.7276
  PR-AUC                   : 0.9695

Confusion Matrix:
  Predicted ->    0 (No)  1 (Yes)
  Actual 0 (No):     198     115
  Actual 1 (Yes):   1287    2877

Classification Report:
                 precision    recall  f1-score   support

Not engaged (0)       0.13      0.63      0.22       313
    Engaged (1)       0.96      0.69      0.80      4164

       accuracy                           0.69      4477
      macro avg       0.55      0.66      0.51      4477
   weighted avg       0.90      0.69      0.76      4477



## 5.2 Train and Tune XGBoost

In [26]:
# XGBoost tuning
xgb_param_grid = {
    'classifier__n_estimators': [100, 200, 300],
    'classifier__max_depth': [3, 5, 7],
    'classifier__learning_rate': [0.01, 0.1, 0.2],
    'classifier__subsample': [0.8, 1.0],
    'classifier__colsample_bytree': [0.8, 1.0],
}

# Compute scale_pos_weight for imbalance handling
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

xgb_tuning_pipeline = Pipeline([
    ('preprocessor', xgb_preprocessor),
    ('classifier', xgb.XGBClassifier(
        scale_pos_weight=scale_pos_weight,
        random_state=RANDOM_STATE,
        eval_metric='logloss'
    ))
])

xgb_grid = GridSearchCV(
    xgb_tuning_pipeline,
    xgb_param_grid,
    cv=ps,
    scoring='f1_macro',
    refit=True,
    n_jobs=-1,
    verbose=0
)

xgb_grid.fit(X_train_val, y_train_val)

xgb_tuned = xgb_grid.best_estimator_
print("XGBoost tuning complete.")
print(f"  Best parameters: {xgb_grid.best_params_}")
print(f"  Best validation F1 (macro): {xgb_grid.best_score_:.4f}")

XGBoost tuning complete.
  Best parameters: {'classifier__colsample_bytree': 1.0, 'classifier__learning_rate': 0.1, 'classifier__max_depth': 7, 'classifier__n_estimators': 300, 'classifier__subsample': 0.8}
  Best validation F1 (macro): 0.5575


In [27]:
# Structured tuning summary for XGBoost
print("=" * 60)
print(" TUNING SUMMARY: XGBoost")
print("=" * 60)
print(f"  Tuning method:          GridSearchCV with PredefinedSplit")
print(f"  Scoring metric:         F1 (macro)")
print(f"  Hyperparameters searched:")
print(f"    - n_estimators:       {xgb_param_grid['classifier__n_estimators']}")
print(f"    - max_depth:          {xgb_param_grid['classifier__max_depth']}")
print(f"    - learning_rate:      {xgb_param_grid['classifier__learning_rate']}")
print(f"    - subsample:          {xgb_param_grid['classifier__subsample']}")
print(f"    - colsample_bytree:   {xgb_param_grid['classifier__colsample_bytree']}")
total_xgb = (len(xgb_param_grid['classifier__n_estimators']) *
             len(xgb_param_grid['classifier__max_depth']) *
             len(xgb_param_grid['classifier__learning_rate']) *
             len(xgb_param_grid['classifier__subsample']) *
             len(xgb_param_grid['classifier__colsample_bytree']))
print(f"  Total configurations:   {total_xgb}")
print(f"  Iterations completed:   {total_xgb}")
print(f"  Best hyperparameters:   {xgb_grid.best_params_}")
print(f"  Best val F1 (macro):    {xgb_grid.best_score_:.4f}")

 TUNING SUMMARY: XGBoost
  Tuning method:          GridSearchCV with PredefinedSplit
  Scoring metric:         F1 (macro)
  Hyperparameters searched:
    - n_estimators:       [100, 200, 300]
    - max_depth:          [3, 5, 7]
    - learning_rate:      [0.01, 0.1, 0.2]
    - subsample:          [0.8, 1.0]
    - colsample_bytree:   [0.8, 1.0]
  Total configurations:   108
  Iterations completed:   108
  Best hyperparameters:   {'classifier__colsample_bytree': 1.0, 'classifier__learning_rate': 0.1, 'classifier__max_depth': 7, 'classifier__n_estimators': 300, 'classifier__subsample': 0.8}
  Best val F1 (macro):    0.5575


In [28]:
# Evaluate tuned XGBoost on validation set
tuned_xgb_val_metrics = evaluate_model(xgb_tuned, X_val, y_val, "Tuned XGBoost", "Validation")


 Tuned XGBoost — Validation Set Results
  Accuracy                 : 0.8099
  Precision (macro)        : 0.6178
  Recall (macro)           : 0.8299
  F1 (macro)               : 0.6366
  ROC-AUC                  : 0.9124
  PR-AUC                   : 0.9928

Confusion Matrix:
  Predicted ->    0 (No)  1 (Yes)
  Actual 0 (No):     267      46
  Actual 1 (Yes):    805    3359

Classification Report:
                 precision    recall  f1-score   support

Not engaged (0)       0.25      0.85      0.39       313
    Engaged (1)       0.99      0.81      0.89      4164

       accuracy                           0.81      4477
      macro avg       0.62      0.83      0.64      4477
   weighted avg       0.93      0.81      0.85      4477



## 5.3 Model Comparison (Test Set)

Now we evaluate all three models on the held-out **test set** for final comparison. This is the first and only time the test set is used for evaluation.

In [29]:
# Evaluate all three models on the TEST set
print("\n" + "#" * 60)
print("# FINAL TEST SET EVALUATION")
print("#" * 60)

models = {
    "Baseline LR": lr_baseline,
    "Tuned LR": lr_tuned,
    "Tuned XGBoost": xgb_tuned,
}

test_results = {}
for name, model in models.items():
    test_results[name] = evaluate_model(model, X_test, y_test, name, "Test")


############################################################
# FINAL TEST SET EVALUATION
############################################################

 Baseline LR — Test Set Results
  Accuracy                 : 0.7151
  Precision (macro)        : 0.5589
  Recall (macro)           : 0.6932
  F1 (macro)               : 0.5355
  ROC-AUC                  : 0.7617
  PR-AUC                   : 0.9724

Confusion Matrix:
  Predicted ->    0 (No)  1 (Yes)
  Actual 0 (No):     209     104
  Actual 1 (Yes):   1172    2993

Classification Report:
                 precision    recall  f1-score   support

Not engaged (0)       0.15      0.67      0.25       313
    Engaged (1)       0.97      0.72      0.82      4165

       accuracy                           0.72      4478
      macro avg       0.56      0.69      0.54      4478
   weighted avg       0.91      0.72      0.78      4478


 Tuned LR — Test Set Results
  Accuracy                 : 0.6838
  Precision (macro)        : 0.5527
  Recall (

In [30]:
# Comparison table
comparison_df = pd.DataFrame(test_results).T
comparison_df = comparison_df.round(4)
print("\n" + "=" * 70)
print(" MODEL COMPARISON — TEST SET")
print("=" * 70)
print(comparison_df.to_string())

# Visual comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart of key metrics
metrics_to_plot = ['F1 (macro)', 'ROC-AUC', 'PR-AUC', 'Accuracy']
comparison_df[metrics_to_plot].plot(kind='bar', ax=axes[0], rot=15, edgecolor='black')
axes[0].set_title('Test Set: Key Metrics by Model', fontsize=12)
axes[0].set_ylabel('Score')
axes[0].set_ylim(0, 1)
axes[0].legend(loc='lower right')

# ROC curves
for name, model in models.items():
    RocCurveDisplay.from_estimator(model, X_test, y_test, name=name, ax=axes[1])
axes[1].set_title('Test Set: ROC Curves', fontsize=12)
axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.5)

plt.tight_layout()
plt.savefig("evidence_claude_code/EDA_claude_code_Pics/model_comparison.png", dpi=150, bbox_inches='tight')
plt.show()


 MODEL COMPARISON — TEST SET
               Accuracy  Precision (macro)  Recall (macro)  F1 (macro)  ROC-AUC  PR-AUC
Baseline LR      0.7151             0.5589          0.6932      0.5355   0.7617  0.9724
Tuned LR         0.6838             0.5527          0.6823      0.5161   0.7331  0.9681
Tuned XGBoost    0.7573             0.5582          0.6686      0.5505   0.7342  0.9705


## 5.4 Final Model Decision

### Model Selection Framework

We use a multi-dimensional, quantitative framework to select the final model. The criteria are weighted to reflect the task context — identifying under-engagement in arts participation for policy purposes.

| Criterion | Weight | Rationale |
|-----------|--------|-----------|
| **F1 (macro)** | 0.30 | Primary performance metric; balances precision and recall across both classes |
| **ROC-AUC** | 0.25 | Overall discrimination ability; robust to threshold choice |
| **PR-AUC** | 0.20 | Particularly relevant for the positive class in imbalanced settings |
| **Recall (macro)** | 0.15 | Important for policy: we want to identify as many under-engaged groups as possible |
| **Interpretability** | 0.10 | Policy stakeholders need to understand and trust the model's decisions |

**Interpretability scoring**: Logistic Regression = 1.0 (fully interpretable coefficients), XGBoost = 0.6 (feature importance available but less transparent).

The final score for each model is the weighted sum of normalised metric values.

In [31]:
# Quantitative model selection
weights = {
    'F1 (macro)': 0.30,
    'ROC-AUC': 0.25,
    'PR-AUC': 0.20,
    'Recall (macro)': 0.15,
    'Interpretability': 0.10,
}

interpretability = {
    'Baseline LR': 1.0,
    'Tuned LR': 1.0,
    'Tuned XGBoost': 0.6,
}

# Build scoring table
scoring_data = {}
for model_name in test_results:
    scoring_data[model_name] = {}
    for criterion, weight in weights.items():
        if criterion == 'Interpretability':
            score = interpretability[model_name]
        else:
            score = test_results[model_name][criterion]
        scoring_data[model_name][criterion] = score

scoring_df = pd.DataFrame(scoring_data).T

# Compute weighted scores
weighted_scores = {}
for model_name in scoring_df.index:
    total = sum(scoring_df.loc[model_name, crit] * w for crit, w in weights.items())
    weighted_scores[model_name] = total

scoring_df['Weighted Score'] = pd.Series(weighted_scores)

print("\n" + "=" * 70)
print(" MODEL SELECTION FRAMEWORK — WEIGHTED SCORING")
print("=" * 70)
print(f"\nWeights: {weights}")
print(f"\nScoring Table:")
print(scoring_df.round(4).to_string())

best_model_name = scoring_df['Weighted Score'].idxmax()
print(f"\n{'*' * 50}")
print(f" SELECTED MODEL: {best_model_name}")
print(f" Weighted Score: {scoring_df.loc[best_model_name, 'Weighted Score']:.4f}")
print(f"{'*' * 50}")


 MODEL SELECTION FRAMEWORK — WEIGHTED SCORING

Weights: {'F1 (macro)': 0.3, 'ROC-AUC': 0.25, 'PR-AUC': 0.2, 'Recall (macro)': 0.15, 'Interpretability': 0.1}

Scoring Table:
               F1 (macro)  ROC-AUC  PR-AUC  Recall (macro)  Interpretability  Weighted Score
Baseline LR        0.5355   0.7617  0.9724          0.6932               1.0          0.7495
Tuned LR           0.5161   0.7331  0.9681          0.6823               1.0          0.7341
Tuned XGBoost      0.5505   0.7342  0.9705          0.6686               0.6          0.7031

**************************************************
 SELECTED MODEL: Baseline LR
 Weighted Score: 0.7495
**************************************************


In [32]:
# Save final comparison and selection results to evidence
comparison_df.to_csv("evidence_claude_code/test_set_comparison.csv")
scoring_df.to_csv("evidence_claude_code/model_selection_scores.csv")
print("Results saved to evidence_claude_code/")
print(f"  - test_set_comparison.csv")
print(f"  - model_selection_scores.csv")

Results saved to evidence_claude_code/
  - test_set_comparison.csv
  - model_selection_scores.csv
